# Customer Segmentation & Campaign Analytics

This notebook demonstrates:
1. Feature engineering for member behavioral segmentation
2. K-Means clustering with optimal K selection
3. Segment profiling and interpretation
4. Campaign performance analysis by segment, channel, and type

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.feature_engineering import compute_derived_features, scale_features, EXTENDED_FEATURES
from src.segmentation import find_optimal_k, fit_kmeans, profile_segments, label_segments
from src.campaign_analytics import (
    campaign_summary_by_segment, campaign_summary_by_channel,
    campaign_summary_by_type, segment_channel_heatmap_data
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Load & Explore Data

In [ ]:
members = pd.read_csv('../data/member_profiles.csv')
campaigns = pd.read_csv('../data/campaign_results.csv')

print(f'Members: {len(members):,}')
print(f'Campaign touches: {len(campaigns):,}')
print(f'\nMember feature summary:')
members.describe().round(1)

## 2. Feature Engineering

In [ ]:
members = compute_derived_features(members)

print('Derived features added:')
print(f'  digital_engagement: portal + mobile sessions (mean: {members["digital_engagement"].mean():.1f})')
print(f'  cost_ratio: member share / total cost (mean: {members["cost_ratio"].mean():.2f})')
print(f'  utilization_intensity: claims + ER + Rx (mean: {members["utilization_intensity"].mean():.1f})')

# Feature correlations
fig, ax = plt.subplots(figsize=(12, 10))
corr = members[EXTENDED_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Optimal K Selection

In [ ]:
X, scaler, features_used = scale_features(members)
k_values, inertias, silhouettes = find_optimal_k(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_values, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].grid(alpha=0.3)

axes[1].plot(k_values, silhouettes, 'rs-', linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score by K', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(alpha=0.3)

best_k = k_values[np.argmax(silhouettes)]
axes[1].axvline(best_k, color='green', linestyle='--', alpha=0.7, label=f'Best K={best_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Best silhouette at K={best_k}: {max(silhouettes):.3f}')
print('Using K=4 for interpretability')

## 4. K-Means Clustering (K=4)

In [ ]:
km, labels, sil_score = fit_kmeans(X, n_clusters=4)
members['segment'] = labels

members, label_map = label_segments(members, features_used)

print(f'Silhouette Score: {sil_score:.3f}')
print(f'\nSegment distribution:')
print(members['segment_label'].value_counts())
print(f'\nLabel mapping: {label_map}')

## 5. Segment Profiling

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

plot_features = [
    ('age', 'Age Distribution'),
    ('digital_engagement', 'Digital Engagement'),
    ('total_cost_ytd', 'Total Cost YTD'),
    ('utilization_intensity', 'Utilization Intensity'),
    ('satisfaction_score', 'Satisfaction Score'),
    ('tenure_months', 'Tenure (Months)'),
]

for idx, (feat, title) in enumerate(plot_features):
    row, col = divmod(idx, 3)
    ax = axes[row][col]
    for label in sorted(members['segment_label'].unique()):
        data = members[members['segment_label'] == label][feat]
        ax.hist(data, bins=20, alpha=0.5, label=label, edgecolor='black')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Segment', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Radar / summary table
summary_cols = ['age', 'digital_engagement', 'total_cost_ytd', 'utilization_intensity',
                'satisfaction_score', 'tenure_months', 'preventive_visits']
segment_means = members.groupby('segment_label')[summary_cols].mean().round(1)
segment_means['count'] = members.groupby('segment_label').size()
segment_means['pct'] = (segment_means['count'] / len(members) * 100).round(1)

print('Segment Profile Summary:')
segment_means

## 6. Campaign Performance by Segment

In [ ]:
seg_campaign = campaign_summary_by_segment(campaigns, members)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

seg_campaign['open_rate'].sort_values().plot(kind='barh', ax=axes[0], color='#3498DB', edgecolor='black')
axes[0].set_title('Open Rate by Segment', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rate')

seg_campaign['click_rate'].sort_values().plot(kind='barh', ax=axes[1], color='#F39C12', edgecolor='black')
axes[1].set_title('Click Rate by Segment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Rate')

seg_campaign['conversion_rate'].sort_values().plot(kind='barh', ax=axes[2], color='#27AE60', edgecolor='black')
axes[2].set_title('Conversion Rate by Segment', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Rate')

plt.tight_layout()
plt.show()

print('\nCampaign Performance by Segment:')
seg_campaign

## 7. Channel Performance

In [ ]:
channel_perf = campaign_summary_by_channel(campaigns)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['open_rate', 'click_rate', 'conversion_rate']
channel_perf[metrics].plot(kind='bar', ax=axes[0], edgecolor='black')
axes[0].set_title('Campaign Metrics by Channel', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Rate')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Metric')

channel_perf['cost_per_conversion'].plot(kind='bar', ax=axes[1], color='#E74C3C', edgecolor='black')
axes[1].set_title('Cost per Conversion by Channel', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cost ($)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('\nChannel Performance:')
channel_perf

## 8. Segment × Channel Heatmap

In [ ]:
heatmap_data = segment_channel_heatmap_data(campaigns, members)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Conversion Rate'})
ax.set_title('Conversion Rate: Segment × Channel', fontsize=14, fontweight='bold')
ax.set_ylabel('Segment')
ax.set_xlabel('Channel')

plt.tight_layout()
plt.show()

## 9. Campaign Type Analysis

In [ ]:
type_perf = campaign_summary_by_type(campaigns)

fig, ax = plt.subplots(figsize=(12, 5))
type_perf[['open_rate', 'click_rate', 'conversion_rate']].plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('Campaign Performance by Type', fontsize=13, fontweight='bold')
ax.set_ylabel('Rate')
ax.tick_params(axis='x', rotation=15)
ax.legend(title='Metric')

plt.tight_layout()
plt.show()

print('\nCampaign Type Performance:')
type_perf

## 10. Key Findings & Recommendations

### Findings
1. **Four distinct segments** emerge from behavioral clustering, each with different engagement patterns and cost profiles
2. **High-Value Engaged members** respond best to campaigns across all channels — they should receive the most investment
3. **Young Digital-First members** respond well to email and SMS but rarely through mail — digital-first strategy is optimal
4. **Cost-Sensitive At-Risk members** have the lowest conversion but highest potential impact — targeted interventions needed
5. **Channel effectiveness varies by segment** — one-size-fits-all campaigns leave value on the table

### Recommendations
1. **Segment-specific campaign strategies** — allocate budget proportional to expected ROI per segment
2. **Digital channel priority for under-45** — shift mail budget to email/SMS for younger segments
3. **At-risk member intervention** — design specialized retention campaigns with personalized offers
4. **A/B test by segment** — move beyond aggregate metrics to segment-level optimization
5. **Automate segment assignment** — integrate clustering into CRM for real-time targeting